DS2002 Midterm: Walmart Hurricane Analytics

Team Landfall Analytics: Russell Fullerton and Todd Balwinski

1. Setup and data loading

In [ ]:
import time
import sqlite3
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_FILES = [
    "walmart_transactions.csv",
    "store_locations.csv",
    "product_catalog.csv",
    "distribution_centers.csv",
    "inventory_and_sales.db",
]
REPO_RAW = "https://raw.githubusercontent.com/toddbalwinski/ds2002-midterm-project/main/data/"


def find_data_dir():
    for folder in (Path("data"), Path("../data"), Path("/content/data"), Path(".")):
        if all((folder / f).exists() for f in DATA_FILES):
            return folder
    target = Path("data")
    target.mkdir(exist_ok=True)
    for f in DATA_FILES:
        if not (target / f).exists():
            print("downloading", f)
            urllib.request.urlretrieve(REPO_RAW + f, target / f)
    return target


DATA_DIR = find_data_dir()
print("data folder:", DATA_DIR.resolve())

raw_tx = pd.read_csv(DATA_DIR / "walmart_transactions.csv", dtype=str, keep_default_na=False)
raw_stores = pd.read_csv(DATA_DIR / "store_locations.csv", dtype=str, keep_default_na=False)
raw_products = pd.read_csv(DATA_DIR / "product_catalog.csv", dtype=str, keep_default_na=False)
raw_dcs = pd.read_csv(DATA_DIR / "distribution_centers.csv", dtype=str, keep_default_na=False)

conn = sqlite3.connect(DATA_DIR / "inventory_and_sales.db")
raw_daily = pd.read_sql_query("SELECT * FROM daily_sales_summary", conn)
raw_inventory = pd.read_sql_query("SELECT * FROM inventory_levels", conn)

for name, df in {"transactions": raw_tx, "stores": raw_stores, "products": raw_products,
                 "distribution centers": raw_dcs, "daily_sales_summary (db)": raw_daily,
                 "inventory_levels (db)": raw_inventory}.items():
    print(f"{name:26s} {df.shape[0]:>6,} rows x {df.shape[1]} cols")

We inspect the raw data

We profile the raw transactions before cleaning them. We look for duplicate rows, inconsistent store and category labels, missing values, and mixed timestamp formats.

In [ ]:
MISSING_TOKENS = {"", "NULL", "NUL", "N/A", "NA", "NAN", "NONE", "NAT", "-"}

def profile(df, name):
    rows = []
    for col in df.columns:
        s = df[col].astype(str).str.strip()
        rows.append({
            "column": col,
            "distinct": df[col].nunique(),
            "blank": int((s == "").sum()),
            "NULL / N/A tokens": int(s.str.upper().isin(MISSING_TOKENS - {""}).sum()),
            "examples": " | ".join(df[col].drop_duplicates().astype(str).head(4)),
        })
    print(f"=== {name}: {len(df):,} rows ===")
    display(pd.DataFrame(rows))

profile(raw_tx, "walmart_transactions.csv")
print("Timestamp formats (digits replaced by 9):")
display(raw_tx["timestamp"].str.replace(r"\d", "9", regex=True).value_counts().to_frame("rows"))
print("Raw category spellings:", raw_tx["category"].nunique())
print("Raw store_id spellings:", sorted(raw_tx["store_id"].unique())[:15], "...")

In [ ]:
display(raw_stores)
display(raw_dcs)
print("daily_sales_summary schema:", conn.execute("SELECT sql FROM sqlite_master WHERE name='daily_sales_summary'").fetchone()[0])
print("inventory_levels schema:   ", conn.execute("SELECT sql FROM sqlite_master WHERE name='inventory_levels'").fetchone()[0])
print("\ndate string lengths in daily_sales_summary:")
print(pd.read_sql_query("SELECT LENGTH(date) AS len, COUNT(*) AS n_rows FROM daily_sales_summary GROUP BY 1", conn))

Our cleaning pipeline

We keep each cleaning step in a function. We record our decisions in one log and use assertions to catch broken joins or unexpected values.

We standardize store IDs to uppercase FL-### values.
We standardize SKUs and map each code to a product family with a mapping table.
We standardize category labels so they match the database categories.

In [ ]:
decision_log = []

def log(file, issue, decision, rows_affected, reason):
    decision_log.append({"file": file, "issue": issue, "decision": decision,
                         "rows_affected": int(rows_affected), "reason": reason})

def report(step, before, after):
    print(f"{step:<48s} rows {before:>6,} -> {after:>6,}")


def std_store_id(s):
    return (s.astype(str).str.strip().str.upper()
             .str.replace("_", "-", regex=False)
             .str.replace(r"^FL-?(\d+)$", r"FL-\1", regex=True))

def std_sku(s):
    return s.astype(str).str.strip().str.upper()

CATEGORY_MAP = {
    "Breakfast Foods": "Breakfast", "Emergency Supplies": "Emergency",
    "Alcoholic Beverages": "Alcohol", "Games & Entertainment": "Entertainment",
    "Groceries": "Grocery", "Household Essentials": "Household",
    "Health & Wellness": "Health", "Baby Products": "Baby",
}
CANONICAL_CATEGORIES = {"Alcohol", "Baby", "Breakfast", "Electronics", "Emergency",
                        "Entertainment", "Grocery", "Health", "Household", "Snacks"}

def std_category(s):
    return s.astype(str).str.strip().str.title().replace(CATEGORY_MAP)

def to_missing(df, file):
    out = df.copy()
    converted = 0
    for col in out.columns:
        if out[col].dtype == object or pd.api.types.is_string_dtype(out[col]):
            s = out[col].astype(str).str.strip()
            mask = s.str.upper().isin(MISSING_TOKENS)
            converted += int(mask.sum())
            out[col] = s.mask(mask)
    log(file, "Missing values written as '', NULL, N/A, nan",
        "Standardized to NaN before any type conversion", converted,
        "A plain isna() count undercounts missing data; every later step needs one definition of missing")
    return out

We build the SKU mapping table

We map the 45 raw SKU codes to 20 product families. We use a left join so unmapped codes remain visible for review.

In [ ]:
SKU_GROUPS = {
    "Strawberry Pop-Tarts": ["PT-12", "SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
    "Bottled Water 24-Pack": ["WTR-24", "WATER24PK", "WTR_24PK"],
    "AA Batteries 4-Pack": ["BATT-100", "BAT-AA4", "BATT_AA_4PK"],
    "LED Flashlight": ["FLASH-01", "LED-FLASH"],
    "Light Beer 24-Pack": ["BEER-24", "LTBEER24"],
    "Monopoly Board Game": ["BG-MONO", "MONOPOLY-01", "GAME-MONO"],
    "Playing Cards": ["CARDS-01", "PLAYCARD-STD"],
    "Streaming Device": ["STREAM-01", "ROKU-STD"],
    "Canned Soup": ["SOUP-CAN", "CNSOUP-01"],
    "White Bread": ["BREAD-WH", "WHITEBREAD"],
    "Peanut Butter 16oz": ["PB-16", "PNUTBTR-16"],
    "Emergency Candles": ["CNDL-EM", "CANDLE-EMRG"],
    "First Aid Kit": ["FAID-01", "FIRSTAID-KIT"],
    "Portable Phone Charger": ["CHRG-PORT", "PORTCHARGER"],
    "Chips Variety Pack": ["CHIPS-VP", "CHIPVARIETY"],
    "Paper Towels": ["PTOWEL-6", "PAPERTOWEL6"],
    "Toilet Paper 12-Roll": ["TP-12", "TOILETPAPER12"],
    "Baby Diapers Size 3": ["DIAP-S3", "DIAPERS-3"],
    "Baby Formula": ["BFORM-01", "BABYFORMULA"],
    "Heavy Duty Tarp": ["TARP-HD", "HDTARP-01"],
}

sku_mapping = pd.DataFrame(
    [(sku, family) for family, skus in SKU_GROUPS.items() for sku in skus],
    columns=["sku", "product_family"],
)
sku_mapping["sku"] = std_sku(sku_mapping["sku"])
assert sku_mapping["sku"].is_unique, "a SKU appears in two families"

def audit_skus(skus, source):
    observed = pd.DataFrame({"sku": std_sku(pd.Series(skus)).unique()})
    joined = observed.merge(sku_mapping, on="sku", how="left", indicator=True, validate="one_to_one")
    unmapped = joined.loc[joined["_merge"] == "left_only", "sku"].tolist()
    print(f"{source:<16s} distinct SKUs {len(observed):>3}  mapped {len(observed) - len(unmapped):>3}  unmapped {unmapped}")
    return unmapped

print(f"mapping table: {len(sku_mapping)} SKU codes -> {sku_mapping['product_family'].nunique()} product families")
_ = audit_skus(raw_products["sku"], "catalog")
_ = audit_skus(raw_tx["sku"], "transactions")
inventory_unmapped = audit_skus(raw_inventory["sku"], "inventory db")
display(sku_mapping[sku_mapping["product_family"] == "Strawberry Pop-Tarts"])

We write the cleaning functions

In [ ]:
TIMESTAMP_FORMATS = [  
    "%Y-%m-%d %H:%M:%S",  
    "%Y-%m-%dT%H:%M:%S",
    "%m/%d/%Y %H:%M",    
    "%m-%d-%Y %H:%M:%S",  
    "%d-%b-%Y %H:%M",   
]

def parse_timestamps(s):
    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for fmt in TIMESTAMP_FORMATS:
        todo = parsed.isna() & s.notna()
        parsed[todo] = pd.to_datetime(s[todo], format=fmt, errors="coerce")
    leftover = parsed.isna() & s.notna()
    assert leftover.sum() == 0, f"unparsed timestamps: {s[leftover].unique()[:5]}"
    return parsed


def clean_transactions(raw):
    F = "walmart_transactions.csv"
    df = to_missing(raw, F)
    n0 = len(df)

    dup = df.duplicated()
    df = df[~dup].copy()
    log(F, "Exact duplicate rows", "Dropped (kept first)", dup.sum(),
        "Same transaction_id and identical fields = the same sale counted twice")
    report("drop exact duplicates", n0, len(df))
    assert df["transaction_id"].is_unique, "duplicate transaction_ids remain after dedup"

    before = df["store_id"].nunique()
    changed = (std_store_id(df["store_id"]) != df["store_id"]).sum()
    df["store_id"] = std_store_id(df["store_id"])
    log(F, f"store_id written {before} ways", "Uppercase, '_'->'-', insert hyphen (FL-239)", changed,
        "Needed to join to store_locations.csv and the database (Q4)")

    df["timestamp"] = parse_timestamps(df["timestamp"])
    missing_ts = df["timestamp"].isna()
    df["date"] = df["timestamp"].dt.normalize()
    df["has_timestamp"] = ~missing_ts
    log(F, "5 timestamp formats (ISO, ISO-T, mm/dd/yyyy, mm-dd-yyyy, dd-Mon-yyyy)",
        "Parsed each format explicitly; any leftover fails an assert", (~missing_ts).sum(),
        "format='mixed' guesses per row; explicit formats cannot silently swap day and month")
    log(F, "Missing timestamp (blank / NULL / N/A)",
        "Kept the rows but flagged has_timestamp=False; excluded from day-level analysis only", missing_ts.sum(),
        "Units still count toward SKU totals (Q2); a date cannot be imputed honestly. Spread checked below")

    df["sku"] = std_sku(df["sku"])
    df = df.merge(sku_mapping, on="sku", how="left", validate="many_to_one")
    log(F, "45 SKU codes for 20 physical products", "Mapped to product_family via the sku_mapping table",
        len(df), "The same product under several SKUs fragments demand (Q2)")

    names_before = df["product_name"].nunique()
    df["product_name_raw"] = df["product_name"]
    df["product_name"] = df["product_family"]
    log(F, f"product_name written {names_before} ways ('POPTART-STRAWBERRY', ' Pop-Tarts Strawberry ')",
        "Replaced with the canonical product_family name; raw kept in product_name_raw", len(df),
        "Free-text names are not a reliable key; the SKU mapping is")

    cats_before = df["category"].nunique()
    df["category"] = std_category(df["category"])
    log(F, f"category written {cats_before} ways", "Title-case + map long labels to the 10 database names",
        len(df), "Must match daily_sales_summary categories for Q5 cross-checks")

    has_dollar = df["unit_price"].astype(str).str.contains(r"\$", regex=True)
    df["unit_price"] = pd.to_numeric(df["unit_price"].str.replace("$", "", regex=False).str.strip(), errors="raise")
    log(F, "unit_price stored as text, some with '$'", "Stripped '$' and converted to float", has_dollar.sum(),
        "Prices must be numeric to compute revenue")

    df["quantity"] = pd.to_numeric(df["quantity"], errors="raise").astype(int)
    neg = df["quantity"] < 0
    df["is_return"] = neg
    df["gross_units"] = df["quantity"].clip(lower=0)
    df["returned_units"] = (-df["quantity"]).clip(lower=0)
    log(F, "Negative quantity", "Kept as returns (is_return); demand uses gross_units, net shown separately",
        neg.sum(), "Small, round values (-1 to -5) look like returns, not typos; dropping them would overstate net sales, "
                   "but counting them against demand would understate what shoppers tried to buy")

    df["revenue"] = df["quantity"] * df["unit_price"]
    return df


TALLAHASSEE_COORDS = (30.4383, -84.2807)

def clean_stores(raw):
    F = "store_locations.csv"
    df = to_missing(raw, F)
    n0 = len(df)
    changed = (std_store_id(df["store_id"]) != df["store_id"]).sum()
    df["store_id"] = std_store_id(df["store_id"])
    log(F, "store_id format (FL239, FL_287)", "Same standardizer as transactions", changed, "Join key consistency")
    dup = df.duplicated(subset="store_id")
    df = df[~dup].copy()
    log(F, "Duplicate store rows after ID fix", "Dropped (rows were identical apart from the ID spelling)", dup.sum(),
        "A duplicated reference row would double every joined transaction")
    report("stores: dedupe on store_id", n0, len(df))

    bad_state = (df["state"] != "FL").sum()
    df["state"] = df["state"].replace({"Florida": "FL", "Fla.": "FL"})
    log(F, "state written FL / Florida / Fla.", "Standardized to FL", bad_state, "Consistency")

    df["latitude"] = pd.to_numeric(df["latitude"])
    df["longitude"] = pd.to_numeric(df["longitude"])
    miss = df["latitude"].isna() | df["longitude"].isna()
    assert set(df.loc[miss, "city"]) <= {"Tallahassee"}, "unexpected store with missing coordinates"
    df.loc[miss, ["latitude", "longitude"]] = TALLAHASSEE_COORDS
    log(F, "Tallahassee (FL-619) latitude/longitude missing",
        "Filled with Tallahassee city-centre coordinates 30.4383, -84.2807", miss.sum(),
        "Other stores use city-centre coordinates; needed for the weather pull and distance-to-track (Q4)")
    df["zip_code"] = df["zip_code"].astype(str).str.zfill(5)
    return df


def clean_products(raw):
    F = "product_catalog.csv"
    df = to_missing(raw, F)
    df["sku"] = std_sku(df["sku"])
    df = df.merge(sku_mapping, on="sku", how="left", validate="one_to_one")
    cats_before = df["category"].nunique()
    df["category"] = std_category(df["category"])
    log(F, f"category written {cats_before} ways", "Same category map as transactions", len(df), "Consistency")
    df["unit_cost"] = pd.to_numeric(df["unit_cost"])
    log(F, "unit_cost missing", "Left as NaN (not used in any question)", df["unit_cost"].isna().sum(),
        "Imputing a cost would invent data we do not need")
    log(F, "package_size inconsistent (4pk, 4-pack, 24 pack) and contradicts names",
        "Not used; flagged as unreliable", len(df),
        "e.g. 'BOTTLED WATER 24PK' listed as '1 pack' -- the column cannot be trusted")
    return df


def clean_dcs(raw):
    F = "distribution_centers.csv"
    df = to_missing(raw, F)
    approx = df["surge_capacity"].astype(str).str.startswith("~")
    df["surge_capacity_is_estimate"] = approx
    df["surge_capacity"] = pd.to_numeric(df["surge_capacity"].str.lstrip("~"))
    log(F, "surge_capacity '~750' stored as text", "Stripped '~', kept 750, flagged surge_capacity_is_estimate", approx.sum(),
        "Keeps the number usable while recording that it is approximate")
    for col in ["avg_daily_shipments", "distance_to_orlando_mi", "cost_per_shipment"]:
        df[col] = pd.to_numeric(df[col])
    log(F, "cost_per_shipment missing (DC-BHM)", "Left as NaN", df["cost_per_shipment"].isna().sum(),
        "No defensible basis to impute a cost")
    return df


def clean_daily_sales(raw):
    F = "inventory_and_sales.db: daily_sales_summary"
    df = raw.copy()
    compact = df["date"].str.fullmatch(r"\d{8}")
    df["date"] = pd.to_datetime(df["date"].where(~compact), format="%Y-%m-%d").fillna(
        pd.to_datetime(df["date"].where(compact), format="%Y%m%d"))
    log(F, "date in two formats (2024-09-12 and 20240912)", "Parsed both explicitly", compact.sum(),
        "pd.to_datetime(errors='coerce') with one format would silently turn these rows into NaT")
    df["store_id"] = std_store_id(df["store_id"])
    df["category"] = std_category(df["category"])
    neg = (df["total_revenue"] < 0) & (df["total_units"] > 0)
    df.loc[neg, "total_revenue"] = df.loc[neg, "total_revenue"].abs()
    log(F, "Negative total_revenue with positive total_units", "Took absolute value", neg.sum(),
        "Units sold cannot produce negative revenue; magnitudes are in the normal range, so it is a sign error")
    assert not df.duplicated(["date", "store_id", "category"]).any()
    return df


def clean_inventory(raw):
    F = "inventory_and_sales.db: inventory_levels"
    df = raw.copy()
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
    df["store_id"] = std_store_id(df["store_id"])
    df["sku"] = std_sku(df["sku"])
    df = df.merge(sku_mapping, on="sku", how="left", validate="many_to_one")
    unm = df["product_family"].isna()
    log(F, f"SKUs not in the catalog: {sorted(df.loc[unm, 'sku'].unique())}",
        "Kept, flagged as unmapped; excluded from family-level inventory", unm.sum(),
        "LEGACY-PT / OLD-BATT-50 look like retired codes and MISC-001 / UNKNOWN-99 are placeholders; "
        "guessing a family could misattribute stock (relevant to Q5)")
    log(F, "units_on_order missing", "Left as NaN (unknown is not zero)", df["units_on_order"].isna().sum(),
        "Filling with 0 would claim no stock was on order")
    return df

We run the cleaning pipeline

In [ ]:
decision_log.clear()
tx = clean_transactions(raw_tx)
stores = clean_stores(raw_stores)
products = clean_products(raw_products)
dcs = clean_dcs(raw_dcs)
daily_db = clean_daily_sales(raw_daily)
inventory = clean_inventory(raw_inventory)

n_before = len(tx)
tx = tx.merge(stores[["store_id", "store_name", "city", "region", "latitude", "longitude"]],
              on="store_id", how="left", validate="many_to_one")
assert len(tx) == n_before, "store join multiplied rows"
log("transactions x stores", "Join on store_id", "Left join, validate='many_to_one'", n_before,
    "Every transaction must keep exactly one store")

sales = tx[tx["has_timestamp"]].copy()
report("analysis table (rows with timestamp)", len(tx), len(sales))

We validate the cleaned data

In [ ]:

assert len(raw_tx) - len(tx) == 30, "expected 30 exact duplicates"
assert tx["transaction_id"].is_unique

assert tx["store_id"].nunique() == 12 and set(tx["store_id"]) == set(stores["store_id"])
assert set(daily_db["store_id"]) == set(stores["store_id"]) and set(inventory["store_id"]) == set(stores["store_id"])
assert tx["city"].notna().all(), "transaction without a store match"

assert set(tx["category"]) == CANONICAL_CATEGORIES and set(daily_db["category"]) == CANONICAL_CATEGORIES
assert tx["product_family"].notna().all(), "unmapped SKU in transactions"
assert tx.groupby("product_family")["category"].nunique().max() == 1, "a product sits in two categories"

assert pd.api.types.is_float_dtype(tx["unit_price"]) and (tx["unit_price"] > 0).all()
assert (sales["date"].dt.year == 2024).all() and (sales["date"].dt.month == 9).all()
assert sales["date"].nunique() == 30
assert stores[["latitude", "longitude"]].notna().all().all()
assert (daily_db["total_revenue"] >= 0).all()
assert daily_db["date"].nunique() == 30 and daily_db["date"].notna().all()

assert tx["quantity"].sum() == raw_tx.drop_duplicates()["quantity"].astype(int).sum()
print("All cleaning assertions passed.")

summary = pd.Series({
    "raw transaction rows": len(raw_tx),
    "after dedup": len(tx),
    "with timestamp (analysis table)": len(sales),
    "missing timestamp (kept, flagged)": int((~tx["has_timestamp"]).sum()),
    "return rows (negative qty)": int(tx["is_return"].sum()),
    "stores": tx["store_id"].nunique(),
    "categories": tx["category"].nunique(),
    "raw SKU codes": tx["sku"].nunique(),
    "product families": tx["product_family"].nunique(),
    "gross units": int(tx["gross_units"].sum()),
    "returned units": int(tx["returned_units"].sum()),
})
display(summary.to_frame("value"))

In [ ]:
LANDFALL = pd.Timestamp("2024-09-26")
BASELINE = (pd.Timestamp("2024-09-01"), pd.Timestamp("2024-09-15"))
SURGE = (pd.Timestamp("2024-09-19"), pd.Timestamp("2024-09-25"))

We check whether missing timestamps are concentrated in a store or category. We compare each group's share of missing timestamps with its share of all rows.

In [ ]:
def missing_share(col):
    t = pd.DataFrame({
        "share_of_all_rows_%": tx[col].value_counts(normalize=True) * 100,
        "share_of_missing_ts_%": tx.loc[~tx["has_timestamp"], col].value_counts(normalize=True) * 100,
    })
    t["ratio"] = t["share_of_missing_ts_%"] / t["share_of_all_rows_%"]
    return t.round(2)

by_cat = missing_share("category")
by_store = missing_share("store_id")
display(by_cat)
print(f"store ratio range: {by_store['ratio'].min():.2f} - {by_store['ratio'].max():.2f}")
print(f"missing rate overall: {(~tx['has_timestamp']).mean():.1%}")

We review the decisions made across all files

In [ ]:
decision_df = pd.DataFrame(decision_log)
decision_df.index = range(1, len(decision_df) + 1)
display(decision_df)
print(len(decision_df), "decisions logged")

We add historical weather

We use the free Open-Meteo Historical Weather API. We pull hourly wind, gust, precipitation, and pressure data for each store. We convert UTC timestamps to America/New_York before grouping by local date. We retry temporary errors and stop if the response is incomplete. We do not invent weather values.

In [ ]:
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY_VARS = ["wind_speed_10m", "wind_gusts_10m", "precipitation", "pressure_msl"]
RETRYABLE = {429, 500, 502, 503, 504}


def fetch_hourly_weather(latitude, longitude, start="2024-08-31", end="2024-10-01", attempts=3):
    params = {"latitude": latitude, "longitude": longitude, "start_date": start, "end_date": end,
              "hourly": ",".join(HOURLY_VARS), "timezone": "GMT", "wind_speed_unit": "kmh"}
    for attempt in range(attempts):
        try:
            r = requests.get(WEATHER_URL, params=params, timeout=30)
            if r.status_code in RETRYABLE and attempt < attempts - 1:
                time.sleep(2 ** attempt); continue
            r.raise_for_status()
            payload = r.json()
            if "hourly" not in payload:
                raise ValueError(f"response missing hourly data: {str(payload)[:200]}")
            hourly = pd.DataFrame(payload["hourly"])
            hourly["time_utc"] = pd.to_datetime(hourly.pop("time"), format="%Y-%m-%dT%H:%M").dt.tz_localize("UTC")
            assert hourly["time_utc"].is_unique
            assert payload.get("timezone") == "GMT", "expected UTC timestamps from the API"
            return hourly
        except requests.RequestException:
            if attempt == attempts - 1:
                raise
            time.sleep(2 ** attempt)


def to_local_daily(hourly):
    h = hourly.copy()
    h["time_local"] = h["time_utc"].dt.tz_convert("America/New_York")
    h["date"] = h["time_local"].dt.tz_localize(None).dt.normalize()
    daily = h.groupby("date").agg(
        wind_max_kmh=("wind_speed_10m", "max"),
        wind_mean_kmh=("wind_speed_10m", "mean"),
        gust_max_kmh=("wind_gusts_10m", "max"),
        precip_mm=("precipitation", "sum"),
        pressure_mean_hpa=("pressure_msl", "mean"),
        pressure_min_hpa=("pressure_msl", "min"),
        hours=("time_utc", "size"),
    ).reset_index()
    daily = daily[(daily["date"] >= "2024-09-01") & (daily["date"] <= "2024-09-30")]
    assert len(daily) == 30 and (daily["hours"] == 24).all(), "incomplete local days"
    return daily.drop(columns="hours")


frames = []
for _, st in stores.iterrows():
    hourly = fetch_hourly_weather(st["latitude"], st["longitude"])
    d = to_local_daily(hourly)
    d.insert(0, "store_id", st["store_id"])
    frames.append(d)
    time.sleep(0.3) 

weather = pd.concat(frames, ignore_index=True)
weather["source"] = "Open-Meteo historical archive (" + WEATHER_URL + ")"
assert not weather.duplicated(["store_id", "date"]).any()
assert len(weather) == 12 * 30
print(f"weather rows: {len(weather)} (12 stores x 30 local days); live API data")

demo = hourly.loc[hourly["time_utc"].between("2024-09-27 02:00", "2024-09-27 05:00", inclusive="both"),
                  ["time_utc", "wind_speed_10m"]].copy()
demo["time_local_eastern"] = demo["time_utc"].dt.tz_convert("America/New_York")
display(demo)
display(weather.head())

We join weather to sales

We aggregate sales to one row per store and day. We join those rows to the matching weather rows and check that the join does not add or remove observations.

In [ ]:
store_day = (sales.groupby(["store_id", "date"])
                  .agg(gross_units=("gross_units", "sum"), transactions=("transaction_id", "size"))
                  .reset_index())
store_day_cat = (sales.pivot_table(index=["store_id", "date"], columns="category",
                                   values="gross_units", aggfunc="sum", fill_value=0)
                      .reset_index())
store_day = store_day.merge(store_day_cat, on=["store_id", "date"], how="left", validate="one_to_one")

n_sales_rows = len(store_day)
sales_weather = store_day.merge(weather, on=["store_id", "date"], how="left",
                                validate="one_to_one", indicator=True)
assert len(sales_weather) == n_sales_rows, "join changed the number of sales rows"
assert (sales_weather["_merge"] == "both").all(), "store-days without weather"
sales_weather = sales_weather.drop(columns="_merge")
assert sales_weather["gross_units"].sum() == sales["gross_units"].sum(), "units changed in the join"
print(f"store-day rows before join: {n_sales_rows}, after: {len(sales_weather)}, unmatched: 0")
display(sales_weather[["store_id", "date", "gross_units", "wind_max_kmh", "precip_mm", "pressure_min_hpa"]].head())

Question 1: We measure demand surges

We compare average daily gross units in the seven days before landfall, September 19 through September 25, with our baseline period, September 1 through September 15. We rank categories by percentage increase and also report the absolute increase.

In [ ]:
def window_avg(df, start, end, by):
    m = df["date"].between(start, end)
    days = (end - start).days + 1
    return df.loc[m].groupby(by)["gross_units"].sum() / days

base = window_avg(sales, *BASELINE, "category")
surge = window_avg(sales, *SURGE, "category")
q1 = pd.DataFrame({"baseline_units_per_day": base, "surge_units_per_day": surge})
q1["abs_increase_per_day"] = q1["surge_units_per_day"] - q1["baseline_units_per_day"]
q1["pct_increase"] = (q1["surge_units_per_day"] / q1["baseline_units_per_day"] - 1) * 100
q1["multiple"] = q1["surge_units_per_day"] / q1["baseline_units_per_day"]
q1["rank_relative"] = q1["pct_increase"].rank(ascending=False).astype(int)
q1["rank_absolute"] = q1["abs_increase_per_day"].rank(ascending=False).astype(int)
q1 = q1.sort_values("pct_increase", ascending=False)
display(q1.round(1))

alt = window_avg(sales, pd.Timestamp("2024-09-08"), pd.Timestamp("2024-09-15"), "category")
alt_pct = ((surge / alt - 1) * 100).sort_values(ascending=False)
print("Top 5 with Sep 1-15 baseline:", list(q1.index[:5]))
print("Top 5 with Sep 8-15 baseline:", list(alt_pct.index[:5]))

tot_base = window_avg(sales, *BASELINE, lambda x: True).iloc[0]
tot_surge = window_avg(sales, *SURGE, lambda x: True).iloc[0]
print(f"All categories: {tot_base:.0f} -> {tot_surge:.0f} units/day ({(tot_surge/tot_base-1)*100:+.0f}%)")
top5 = list(q1.index[:5])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
colors = ["tab:orange" if c in top5 else "tab:gray" for c in q1.index]
axes[0].barh(q1.index[::-1], q1["pct_increase"][::-1], color=colors[::-1])
for y, (c, v) in enumerate(q1["pct_increase"][::-1].items()):
    axes[0].text(v + 5, y, f"+{v:.0f}%", va="center", fontsize=9)
axes[0].set_title("Breakfast (Pop-Tarts) led the relative surge")
axes[0].set_xlabel("% increase, avg daily units\nSurge Sep 19-25 vs baseline Sep 1-15")
axes[0].set_xlim(0, q1["pct_increase"].max() * 1.18)

q1_abs = q1.sort_values("abs_increase_per_day")
axes[1].barh(q1_abs.index, q1_abs["abs_increase_per_day"], color="tab:blue")
axes[1].set_title("Emergency supplies added the most units")
axes[1].set_xlabel("Extra units per day (surge minus baseline)")
plt.suptitle("Relative and absolute surge by category", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
cat_daily = sales.pivot_table(index="date", columns="category", values="gross_units", aggfunc="sum")
fig, ax = plt.subplots(figsize=(11.5, 5))
ax.axvspan(*SURGE, color="tab:orange", alpha=0.15, label="Surge window")
for c in top5:
    ax.plot(cat_daily.index, cat_daily[c], marker="o", ms=3.5, label=c)
ax.axvline(LANDFALL, color="tab:red", ls="--", label="Landfall")
ax.set_title("Q1: Top 5 surging categories all peak Sep 24-25, two days before landfall")
ax.set_xlabel("Date (September 2024)")
ax.set_ylabel("Units sold per day (all stores)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(ncol=2)
plt.tight_layout(); plt.show()

In [ ]:
q1_peak = (cat_daily.loc["2024-09-25"] / cat_daily.loc[BASELINE[0]:BASELINE[1]].mean() - 1) * 100
print("Peak day (Sep 25) vs baseline, % increase:")
print(q1_peak.sort_values(ascending=False).round(0).to_string())

Our Question 1 write-up

We used the table and charts to identify the top five categories. We reported percentage and absolute increases, explained our date windows, and noted any ranking changes in the sensitivity check.

Question 2: We consolidate Pop-Tart SKUs

We compare the fragmented raw SKU view with the consolidated product-family view. We check that consolidation changes labels only and does not change unit totals.

In [ ]:
pt = sales[sales["product_family"] == "Strawberry Pop-Tarts"]
frag = pt.pivot_table(index="date", columns="sku", values="gross_units", aggfunc="sum", fill_value=0)
cons = pt.groupby("date")["gross_units"].sum().rename("consolidated")

assert frag.values.sum() == cons.sum()
by_sku_all = tx.groupby("sku")["gross_units"].sum()
by_family_all = tx.groupby("product_family")["gross_units"].sum()
assert by_sku_all.sum() == by_family_all.sum() == tx["gross_units"].sum()
assert by_family_all["Strawberry Pop-Tarts"] == by_sku_all[SKU_GROUPS["Strawberry Pop-Tarts"]].sum()
print("Unit totals identical before and after consolidation:", int(by_family_all.sum()))

def wavg(series, window):
    return series.loc[window[0]:window[1]].mean()

q2 = pd.DataFrame({
    "baseline/day": [wavg(frag[c], BASELINE) for c in frag] + [wavg(cons, BASELINE)],
    "surge/day": [wavg(frag[c], SURGE) for c in frag] + [wavg(cons, SURGE)],
    "peak day (Sep 25)": [frag.loc["2024-09-25", c] for c in frag] + [cons.loc["2024-09-25"]],
}, index=list(frag.columns) + ["CONSOLIDATED"])
q2["peak multiple vs baseline"] = q2["peak day (Sep 25)"] / q2["baseline/day"]
display(q2.round(1))

print(f"True daily Pop-Tart volume: {cons.mean():.0f} units/day over September "
      f"(baseline {wavg(cons, BASELINE):.0f}/day, surge {wavg(cons, SURGE):.0f}/day, peak {cons.max():.0f} on {cons.idxmax():%b %d})")
print(f"Largest single SKU in the surge window carries only {q2.iloc[:-1]['surge/day'].max() / q2.loc['CONSOLIDATED','surge/day']:.0%} of true demand")

In [ ]:
surge_rows = sales[sales["date"].between(*SURGE)]
rank_sku = surge_rows.groupby("sku")["gross_units"].sum().rank(ascending=False, method="min")
rank_family = surge_rows.groupby("product_family")["gross_units"].sum().rank(ascending=False, method="min")
best_pt_sku_rank = int(rank_sku[SKU_GROUPS["Strawberry Pop-Tarts"]].min())
print(f"Surge week rank -- best single Pop-Tart SKU: #{best_pt_sku_rank} of {len(rank_sku)} SKU codes; "
      f"consolidated Pop-Tarts: #{int(rank_family['Strawberry Pop-Tarts'])} of {len(rank_family)} products")

fam_counts = sku_mapping["product_family"].value_counts()
print(f"Fragmentation across the catalog: {len(sku_mapping)} SKU codes -> {len(fam_counts)} products; "
      f"{(fam_counts > 1).sum()} products are split; largest family: {fam_counts.idxmax()} ({fam_counts.max()} codes)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
for c in frag.columns:
    axes[0].plot(frag.index, frag[c], marker="o", ms=3, label=c)
axes[0].set_title("Fragmented view: four 'separate' products")
axes[1].plot(cons.index, cons, color="tab:red", marker="o", ms=3, lw=2.2, label="Strawberry Pop-Tarts (all 4 SKUs)")
axes[1].axhline(wavg(cons, BASELINE), color="gray", ls=":", label=f"Baseline {wavg(cons, BASELINE):.0f}/day")
axes[1].set_title("Consolidated view: one product")
for ax in axes:
    ax.axvspan(*SURGE, color="tab:orange", alpha=0.15)
    ax.axvline(LANDFALL, color="tab:red", ls="--", lw=1)
    ax.set_xlabel("Date (September 2024)")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.legend(fontsize=8)
axes[0].set_ylabel("Units sold per day (all stores)")
plt.suptitle("Consolidation reveals the real Pop-Tart surge is about 4x larger than any single SKU shows",
             fontweight="bold")
plt.tight_layout(); plt.show()

Our Question 2 write-up

We reported true daily Pop-Tart volume, compared the fragmented and consolidated views, and explained how the two views could change replenishment, ranking, and pre-positioning decisions.

Question 3: We compare sales with weather

We compare store-day sales with local wind, precipitation, and pressure. We use Pearson correlations and a lag analysis. We report the limits of using one storm and 30 days of statewide data.

In [ ]:
WEATHER_COLS = ["wind_max_kmh", "gust_max_kmh", "precip_mm", "pressure_min_hpa"]
CATS = sorted(CANONICAL_CATEGORIES)

corr = sales_weather[["gross_units"] + CATS + WEATHER_COLS].corr(method="pearson")
q3_corr = corr.loc[["gross_units"] + CATS, WEATHER_COLS]
print(f"Correlation over n = {len(sales_weather)} store-days")
display(q3_corr.round(2))

fig, ax = plt.subplots(figsize=(7.5, 6.5))
image = ax.imshow(q3_corr.to_numpy(), cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(WEATHER_COLS)), WEATHER_COLS, rotation=35, ha="right")
ax.set_yticks(range(len(q3_corr.index)), q3_corr.index)
for row in range(q3_corr.shape[0]):
    for col in range(q3_corr.shape[1]):
        ax.text(col, row, f"{q3_corr.iloc[row, col]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Pearson r")
ax.set_title(f"Same day correlation of sales with local weather\n(store-days, n = {len(sales_weather)})")
ax.set_xlabel("Weather indicator (Open-Meteo, local day)")
ax.set_ylabel("Units sold (total and by category)")
plt.tight_layout(); plt.show()

We test for a sales-weather lag

We shift weather by k days and compare it with sales on day t. A positive k means sales come before the weather. A negative k means sales follow the weather. We use the statewide daily series for this test.

In [ ]:
state = (sales_weather.groupby("date")
         .agg(gross_units=("gross_units", "sum"), wind_max_kmh=("wind_max_kmh", "mean"),
              gust_max_kmh=("gust_max_kmh", "mean"), precip_mm=("precip_mm", "mean"),
              pressure_min_hpa=("pressure_min_hpa", "mean"))
         .sort_index())
for c in CATS:
    state[c] = sales_weather.groupby("date")[c].sum()

LAGS = range(-4, 5)
lag_rows = []
for var in WEATHER_COLS:
    for k in LAGS:
        pair = pd.concat([state["gross_units"], state[var].shift(-k)], axis=1).dropna()
        lag_rows.append({"weather": var, "lag_days": k, "r": pair.corr().iloc[0, 1], "n_days": len(pair)})
lag_df = pd.DataFrame(lag_rows)
lag_table = lag_df.pivot(index="lag_days", columns="weather", values="r")
display(lag_table.round(2))

best = lag_df.loc[lag_df.groupby("weather")["r"].apply(lambda s: s.abs().idxmax())]
display(best.round(2))

peak_sales = state["gross_units"].idxmax()
peak_wind = state["gust_max_kmh"].idxmax()
low_pres = state["pressure_min_hpa"].idxmin()
peak_rain = state["precip_mm"].idxmax()
print(f"Sales peak: {peak_sales:%b %d} | gust peak: {peak_wind:%b %d} | rain peak: {peak_rain:%b %d} | "
      f"pressure low: {low_pres:%b %d} | sales lead the wind peak by {(peak_wind - peak_sales).days} days")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
for var, style in zip(WEATHER_COLS, ["-o", "-s", "-^", "-D"]):
    sub = lag_df[lag_df["weather"] == var]
    ax.plot(sub["lag_days"], sub["r"], style, label=var)
ax.axhline(0, color="black", lw=0.8)
ax.axvline(0, color="gray", ls=":")
ax.set_title("Sales correlate best with weather that has not arrived yet\n"
             "(statewide daily units vs weather shifted by k days, n = 26-30 days)")
ax.set_xlabel("k (days): positive = weather k days AFTER the sales day")
ax.set_ylabel("Pearson r")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

fig, ax1 = plt.subplots(figsize=(11.5, 4.8))
ax1.bar(state.index, state["gross_units"], color="tab:blue", alpha=0.55, label="Units sold (all stores)")
ax1.set_ylabel("Units sold per day", color="tab:blue")
ax2 = ax1.twinx()
ax2.plot(state.index, state["gust_max_kmh"], color="tab:red", marker="o", ms=3, label="Max wind gust (avg of stores)")
ax2.plot(state.index, state["precip_mm"] * 2, color="tab:green", marker="^", ms=3, label="Precipitation x2 (mm)")
ax2.set_ylabel("Gust (km/h) / precipitation x2 (mm)")
ax1.axvline(LANDFALL, color="black", ls="--", lw=1)
ax1.set_title("Buying peaked before the wind and rain arrived")
ax1.set_xlabel("Date (September 2024)")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
same_day = q3_corr.loc["gross_units"]
print("Same-day r, total units vs:", same_day.round(2).to_dict())
for _, row in best.iterrows():
    direction = "AHEAD of (forecast-driven)" if row["lag_days"] > 0 else ("same day as" if row["lag_days"] == 0 else "AFTER (reactive to)")
    print(f"  {row['weather']:<17s} strongest r = {row['r']:+.2f} at k = {row['lag_days']:+d} -> sales move {direction} the weather (n = {row['n_days']} days)")

Our Question 3 write-up

We used the correlation heatmap, lag table, lag plot, and sales-weather chart to describe the relationships among sales, wind, precipitation, and pressure. We explained the lag convention and the limits of our data.

Question 4: We compare stores by location

We use the historical Helene track from the National Hurricane Center. For each store, we calculate distance to the track, surge size, onset day, and peak day. We use the map, table, scatterplot, and heatmap to compare stores.

In [ ]:
TRACK = pd.DataFrame({
    "time_utc": pd.to_datetime(["2024-09-26 03:00", "2024-09-26 06:00", "2024-09-26 09:00", "2024-09-26 15:00",
                                "2024-09-26 18:00", "2024-09-26 21:00", "2024-09-27 00:00", "2024-09-27 03:00",
                                "2024-09-27 09:00"]),
    "lat": [23.1, 23.7, 24.2, 25.5, 26.4, 27.9, 28.7, 29.9, 32.8],
    "lon": [-86.6, -86.5, -86.2, -85.5, -85.0, -84.6, -84.3, -83.9, -83.0],
})

def haversine_mi(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))

dense = []
for (_, a), (_, b) in zip(TRACK.iloc[:-1].iterrows(), TRACK.iloc[1:].iterrows()):
    t = np.linspace(0, 1, 300, endpoint=False)
    dense.append(pd.DataFrame({"lat": a.lat + t * (b.lat - a.lat), "lon": a.lon + t * (b.lon - a.lon)}))
dense = pd.concat(dense + [TRACK[["lat", "lon"]].tail(1)], ignore_index=True)

LANDFALL_PT = (29.9, -83.9)
geo = stores[["store_id", "city", "region", "latitude", "longitude"]].copy()
geo["dist_to_track_mi"] = [haversine_mi(r.latitude, r.longitude, dense["lat"].values, dense["lon"].values).min()
                           for r in geo.itertuples()]
geo["dist_to_landfall_mi"] = haversine_mi(geo["latitude"], geo["longitude"], *LANDFALL_PT)

sd = sales.pivot_table(index="date", columns="store_id", values="gross_units", aggfunc="sum")
base_s = sd.loc[BASELINE[0]:BASELINE[1]].mean()
surge_s = sd.loc[SURGE[0]:SURGE[1]].mean()
ratio = sd / base_s

def onset(col, thresh=1.5):
    above = ratio[col] >= thresh
    run = above & above.shift(-1, fill_value=True) 
    return run[run].index.min()

geo = geo.set_index("store_id")
geo["baseline_units_per_day"] = base_s
geo["surge_units_per_day"] = surge_s
geo["surge_multiple"] = surge_s / base_s
geo["onset_day"] = [onset(s) for s in geo.index]
geo["peak_day"] = sd.idxmax()
geo["peak_multiple"] = sd.max() / base_s
z = (geo["surge_multiple"] - geo["surge_multiple"].mean()) / geo["surge_multiple"].std()
geo["surge_z"] = z
geo = geo.sort_values("dist_to_track_mi")
display(geo.drop(columns=["latitude", "longitude"]).round({c: 2 for c in geo.select_dtypes("number").columns}))

r_dist = geo["dist_to_track_mi"].corr(geo["surge_multiple"])
r_peak = geo["dist_to_track_mi"].corr(geo["peak_multiple"])
print(f"corr(distance to track, surge multiple) = {r_dist:+.2f}  (n = 12 stores)")
print(f"corr(distance to track, peak multiple)  = {r_peak:+.2f}")
print(f"surge multiple range: {geo['surge_multiple'].min():.2f}x - {geo['surge_multiple'].max():.2f}x; "
      f"onset days: {sorted(geo['onset_day'].dt.strftime('%b %d').unique())}; "
      f"peak days: {sorted(geo['peak_day'].dt.strftime('%b %d').unique())}")
outliers = geo[geo["surge_z"].abs() >= 1.5]
print("Stores with |z| >= 1.5 on surge multiple:", outliers.index.tolist())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), gridspec_kw={"width_ratios": [1, 1.25]})
ax = axes[0]
ax.plot(TRACK["lon"], TRACK["lat"], "-o", color="tab:red", ms=3, label="Helene track (NHC)")
ax.scatter(*LANDFALL_PT[::-1], marker="*", s=250, color="darkred", zorder=5, label="Landfall")
sc = ax.scatter(geo["longitude"], geo["latitude"], c=geo["surge_multiple"], s=120, cmap="YlOrRd",
                edgecolor="black", zorder=4)
for sid, r in geo.iterrows():
    ax.annotate(f"{r.city}", (r.longitude, r.latitude), xytext=(4, 4), textcoords="offset points", fontsize=7.5)
plt.colorbar(sc, ax=ax, label="Surge multiple (surge / baseline)")
ax.set_xlim(-88, -79.5); ax.set_ylim(22.8, 31.2)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("Stores and Helene's track")
ax.legend(loc="lower left", fontsize=8)

ax = axes[1]
ax.scatter(geo["dist_to_track_mi"], geo["surge_multiple"], s=80, color="tab:blue")
for sid, r in geo.iterrows():
    ax.annotate(f"{sid} {r.city}", (r.dist_to_track_mi, r.surge_multiple), xytext=(4, 3),
                textcoords="offset points", fontsize=7.5)
m, b = np.polyfit(geo["dist_to_track_mi"], geo["surge_multiple"], 1)
xs = np.linspace(0, geo["dist_to_track_mi"].max() * 1.05, 50)
ax.plot(xs, m * xs + b, ls="--", color="gray", label=f"linear fit, r = {r_dist:+.2f}")
ax.set_xlabel("Minimum distance to Helene's track (miles)")
ax.set_ylabel("Surge multiple (Sep 19-25 vs Sep 1-15)")
ax.set_title("Distance to the track barely predicts surge size")
ax.legend()
plt.suptitle("Every store surged by a similar multiple whether near the path or not", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
order = geo.index.tolist() 
heat = ratio[order].T
heat.columns = heat.columns.strftime("%d")
fig, ax = plt.subplots(figsize=(13, 5))
image = ax.imshow(heat.to_numpy(), cmap="YlOrRd", aspect="auto", vmin=0.5, vmax=4)
ax.set_xticks(range(len(heat.columns)), heat.columns)
ax.set_yticks(range(len(heat.index)), [f"{s} {geo.loc[s, 'city']} ({geo.loc[s, 'dist_to_track_mi']:.0f} mi)" for s in order])
fig.colorbar(image, ax=ax, label="Units / store's own baseline")
ax.set_title("Surge timing is nearly identical across stores")
ax.set_xlabel("Day of September 2024"); ax.set_ylabel("")
plt.tight_layout(); plt.show()

Our Question 4 write-up

We used the track map, store table, scatterplot, and heatmap to assess whether proximity predicted surge size or timing. We reported the distance correlations, onset and peak definitions, and outliers.

Question 5: We investigate the board-game signal

We compare the POS transactions with both SQLite tables. We check peak versus weekly lift, category composition, SKU miscoding, inventory changes, reporting lag, and POS-to-database differences.

In [ ]:
for tbl in ["daily_sales_summary", "inventory_levels"]:
    print(tbl, [r[1] for r in conn.execute(f"PRAGMA table_info({tbl})")])

q5_sql = '''
WITH d AS (
  SELECT CASE WHEN LENGTH(date) = 8
              THEN SUBSTR(date,1,4) || '-' || SUBSTR(date,5,2) || '-' || SUBSTR(date,7,2)
              ELSE date END AS day,
         category, total_units
  FROM daily_sales_summary
)
SELECT day,
       SUM(CASE WHEN category = 'Entertainment' THEN total_units END) AS ent_units,
       SUM(total_units) AS all_units
FROM d
GROUP BY day
ORDER BY day
'''
db_ent = pd.read_sql_query(q5_sql, conn, parse_dates=["day"]).set_index("day")
db_ent["ent_share_%"] = db_ent["ent_units"] / db_ent["all_units"] * 100
assert len(db_ent) == 30, "SQL date normalization lost days"

pos_ent = pd.DataFrame({
    "ent_units": cat_daily["Entertainment"],
    "all_units": cat_daily.sum(axis=1),
})
pos_ent["ent_share_%"] = pos_ent["ent_units"] / pos_ent["all_units"] * 100

def lift(s, kind="window"):
    b = s.loc[BASELINE[0]:BASELINE[1]].mean()
    return (s.loc[SURGE[0]:SURGE[1]].mean() / b - 1) * 100 if kind == "window" else (s.max() / b - 1) * 100

q5_lift = pd.DataFrame({
    "POS csv": [lift(pos_ent["ent_units"], "peak"), lift(pos_ent["ent_units"]), lift(pos_ent["all_units"])],
    "SQLite db": [lift(db_ent["ent_units"], "peak"), lift(db_ent["ent_units"]), lift(db_ent["all_units"])],
}, index=["Entertainment: PEAK day vs baseline %", "Entertainment: surge-week avg vs baseline %",
          "ALL categories: surge-week avg vs baseline %"])
display(q5_lift.round(0))

share = pd.DataFrame({
    "POS share %": [pos_ent["ent_share_%"].loc[BASELINE[0]:BASELINE[1]].mean(), pos_ent["ent_share_%"].loc[SURGE[0]:SURGE[1]].mean()],
    "DB share %": [db_ent["ent_share_%"].loc[BASELINE[0]:BASELINE[1]].mean(), db_ent["ent_share_%"].loc[SURGE[0]:SURGE[1]].mean()],
}, index=["baseline Sep 1-15", "surge Sep 19-25"])
display(share.round(2))
print("Entertainment's rank among 10 categories by peak-day lift (POS):",
      int(q1_peak.rank(ascending=False)["Entertainment"]))

In [ ]:
ent = sales[sales["category"] == "Entertainment"]
raw_labels = raw_tx.loc[std_category(raw_tx["category"]) == "Entertainment", "category"].value_counts()
print("Raw labels lumped into Entertainment:", raw_labels.to_dict())
fam = ent.pivot_table(index="date", columns="product_family", values="gross_units", aggfunc="sum")
fam_lift = pd.DataFrame({
    "baseline/day": fam.loc[BASELINE[0]:BASELINE[1]].mean(),
    "surge/day": fam.loc[SURGE[0]:SURGE[1]].mean(),
})
fam_lift["surge %"] = (fam_lift["surge/day"] / fam_lift["baseline/day"] - 1) * 100
fam_lift["peak %"] = (fam.max() / fam_lift["baseline/day"] - 1) * 100
display(fam_lift.round(0))

print("Categories seen per Entertainment product:", ent.groupby("product_family")["category"].unique().to_dict())
print("Non-Entertainment rows carrying a board-game SKU:",
      int(((tx["product_family"] == "Monopoly Board Game") & (tx["category"] != "Entertainment")).sum()))

upt = ent.assign(window=np.select([ent["date"].between(*BASELINE), ent["date"].between(*SURGE)], ["baseline", "surge"], "other"))
print(upt[upt["window"] != "other"].groupby("window").agg(transactions_per_day=("transaction_id", lambda s: s.size),
      units_per_txn=("gross_units", "mean")).assign(
      transactions_per_day=lambda d: d["transactions_per_day"] / np.where(d.index == "baseline", 15, 7)).round(2))

In [ ]:
inv_sql = '''
SELECT date,
       SUM(CASE WHEN sku IN ('BG-MONO','MONOPOLY-01','GAME-MONO') THEN units_on_hand END) AS boardgame_on_hand,
       SUM(CASE WHEN sku IN ('CARDS-01','PLAYCARD-STD') THEN units_on_hand END)           AS cards_on_hand,
       SUM(CASE WHEN sku IN ('MISC-001','UNKNOWN-99','LEGACY-PT','OLD-BATT-50') THEN units_on_hand END) AS unmapped_on_hand,
       COUNT(*) AS snapshot_rows
FROM inventory_levels
GROUP BY date
ORDER BY date
'''
inv = pd.read_sql_query(inv_sql, conn, parse_dates=["date"]).set_index("date")
print("Inventory snapshot rows per day:", inv["snapshot_rows"].unique(),
      "(144 = 12 stores x 12 SKUs sampled per day, so each SKU is only observed on some days)")
inv_sum = pd.DataFrame({
    "baseline avg on hand": inv.loc[BASELINE[0]:BASELINE[1]].mean(),
    "surge avg on hand": inv.loc[SURGE[0]:SURGE[1]].mean(),
}).drop("snapshot_rows")
inv_sum["change %"] = (inv_sum["surge avg on hand"] / inv_sum["baseline avg on hand"] - 1) * 100
display(inv_sum.round(0))

bg_sales = fam["Monopoly Board Game"]
bg_drop = -inv["boardgame_on_hand"].diff()
lag_inv = {k: bg_sales.corr(bg_drop.shift(-k)) for k in range(0, 4)}
print("corr(board-game sales[t], inventory drop[t+k]):", {k: round(v, 2) for k, v in lag_inv.items()})

recon = pd.DataFrame({"POS units": pos_ent["ent_units"], "DB units": db_ent["ent_units"]})
recon["DB / POS"] = recon["DB units"] / recon["POS units"]
print(f"DB/POS ratio for Entertainment: mean {recon['DB / POS'].mean():.1f}x, "
      f"range {recon['DB / POS'].min():.1f}x - {recon['DB / POS'].max():.1f}x  (the two sources do not reconcile in level)")

db_cat = daily_db.pivot_table(index="date", columns="category", values="total_units", aggfunc="sum")
win = lambda df, w: df.loc[w[0]:w[1]].mean()
artifact = pd.DataFrame({
    "POS surge lift %": (win(cat_daily, SURGE) / win(cat_daily, BASELINE) - 1) * 100,
    "DB surge lift %": (win(db_cat, SURGE) / win(db_cat, BASELINE) - 1) * 100,
    "DB/POS ratio baseline": win(db_cat / cat_daily, BASELINE),
    "DB/POS ratio surge": win(db_cat / cat_daily, SURGE),
})
artifact["DB minus POS lift (pts)"] = artifact["DB surge lift %"] - artifact["POS surge lift %"]
artifact["DB/POS ratio change %"] = (artifact["DB/POS ratio surge"] / artifact["DB/POS ratio baseline"] - 1) * 100
artifact = artifact.sort_values("DB/POS ratio change %", ascending=False)
display(artifact.round(2))
print("Categories where the DB reports a BIGGER surge than the POS file:",
      artifact.index[artifact["DB minus POS lift (pts)"] > 0].tolist())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
ax = axes[0]
ax.plot(pos_ent.index, pos_ent["ent_units"] / pos_ent["ent_units"].loc[BASELINE[0]:BASELINE[1]].mean() * 100,
        label="Entertainment", color="tab:purple", lw=2)
ax.plot(pos_ent.index, pos_ent["all_units"] / pos_ent["all_units"].loc[BASELINE[0]:BASELINE[1]].mean() * 100,
        label="All categories", color="black", ls="--")
ax.axhline(350, color="tab:red", ls=":", lw=1, label="+250% line")
ax.set_title("Entertainment moves with total traffic")
ax.set_ylabel("Index (baseline = 100)")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(pos_ent.index, pos_ent["ent_share_%"], label="POS csv", marker="o", ms=3)
ax.plot(db_ent.index, db_ent["ent_share_%"], label="SQLite db", marker="s", ms=3)
ax.set_ylim(0, 15)
ax.set_title("Share of units: POS falls, DB rises")
ax.set_ylabel("Entertainment share of all units (%)")
ax.legend(fontsize=8)

ax = axes[2]
for c in fam.columns:
    ax.plot(fam.index, fam[c], marker="o", ms=3, label=c)
ax.set_title("Lumping: board games + playing cards")
ax.set_ylabel("Units sold per day (POS)")
ax.legend(fontsize=8)
for ax in axes:
    ax.axvspan(*SURGE, color="tab:orange", alpha=0.15)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.set_xlabel("Date (September 2024)")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("The plus 250% is a peak day figure, at the register Entertainment grew slower than the store",
             fontweight="bold")
plt.tight_layout(); plt.show()

Our Question 5 write-up

We used the POS and SQLite outputs to decide whether the reported Entertainment increase was a real board-game signal or a data artifact. We discussed peak versus weekly lift, traffic share, product composition, SKU miscoding, inventory changes, sampled snapshots, and POS-to-database differences. We supported our stocking recommendation with the evidence.

In [ ]:
conn.close()